In [1]:
import pandas as pd

noblocks = pd.read_csv("minhashblocksample_noblocks.lite.csv").rename(columns={"score": "noblocks_score"})
blocks = pd.read_csv("minhashblocksample_blocks.lite.csv").rename(columns={"score": "blocks_score"})
D = blocks.merge(noblocks, on=["doc_id", "method"])[["blocks_score", "noblocks_score", "doc_id", "method"]].drop_duplicates()

hardness = blocks.merge(noblocks, on=["doc_id", "method"])[["blocks_score", "noblocks_score", "doc_id", "method"]].drop_duplicates()
hardness = hardness[hardness["method"] == "loss"]
hardness["hardness"] = (hardness["blocks_score"] + hardness["noblocks_score"])/2
doc2loss = {k:v for k, v in zip(hardness["doc_id"],hardness["hardness"])}
D["delta"] = D["blocks_score"] - D["noblocks_score"]
D["hardness"] = D["doc_id"].apply(lambda x: doc2loss[x])

len(D)

30000

In [3]:
D[["method", "delta"]].groupby(["method"]).mean().reset_index()

,method,delta
0,loss,-0.022348
1,min_k,-0.068577
2,zlib,0.000006


In [3]:
from scipy.stats import mannwhitneyu

x = D[D["method"] == "loss"]["blocks_score"].to_list()
y = D[D["method"] == "loss"]["noblocks_score"].to_list()

stat, p = mannwhitneyu(x, y, alternative='greater')  # or 'less', 'greater'
print(f"Mann–Whitney U statistic={stat}, p-value={p}")

Mann–Whitney U statistic=50461959.0, p-value=0.1289151328821147


In [48]:
import numpy as np
#Z = D[D["method"] == "loss"]
#for _, z in Z[Z["doc_id"].apply(lambda x: "/local/" in x)].sort_values("delta", ascending=False)[["doc_id", "delta"]].iterrows():
#    print(z['doc_id'], z['delta'])

In [69]:
Z = D[D["method"] == "loss"]
Z = Z[Z["doc_id"].apply(lambda x: "/local/" in x)].copy()
Z["delta"].mean()

-0.05769309680448693

In [70]:
len(Z)

181

In [72]:
! wc -l minhashblocksample_blocks.lite.csv

   60001 minhashblocksample_blocks.lite.csv


In [21]:
import pandas as pd

stats = pd.read_csv("stats.csv")
noblocks = pd.read_csv("minhashblocksample_noblocks.lite.csv").rename(columns={"score": "noblocks_score"})
noblocks = stats.merge(noblocks, left_on='url', right_on="doc_id").drop(columns=['size'])
stats = pd.read_csv("stats.csv")
blocks = pd.read_csv("minhashblocksample_blocks.lite.csv").rename(columns={"score": "blocks_score"})
blocks = stats.merge(blocks, left_on='url', right_on="doc_id")

In [91]:
import pandas as pd

stats = pd.read_csv("stats.csv")
noblocks = pd.read_csv("minhashblocksample_noblocks.lite.csv").rename(columns={"score": "noblocks_score"})
noblocks = stats.merge(noblocks, left_on='url', right_on="doc_id").drop(columns=['size'])
stats = pd.read_csv("stats.csv")
blocks = pd.read_csv("minhashblocksample_blocks.lite.csv").rename(columns={"score": "blocks_score"})
blocks = stats.merge(blocks, left_on='url', right_on="doc_id")

method = "loss"
D = blocks.merge(noblocks, on=["doc_id", "method"])[["blocks_score", "size", "noblocks_score", "doc_id", "method"]].drop_duplicates()
D["delta"] = D["blocks_score"] - D["noblocks_score"]
D = D[D["size"] <= 30].copy()
D["size_bin"] = pd.cut(D["size"], bins=range(0, 31, 3), right=True)
D = D[D['method'] == method].copy()

df = D.groupby("size_bin").agg(
    delta_mean=("delta", "mean"),
    count=("delta", "count")
).reset_index()

import altair as alt

df["size_bin"] = df["size_bin"].astype(str)
alt.Chart(df).mark_bar().encode(
    x=alt.X("size_bin:N", sort=None, title="Count of similar documents in BlockBench"),
    y=alt.Y("delta_mean:Q", title="Mean ATE"),
    tooltip=["size_bin", "delta_mean", "count"]
).properties(
    title=f"Mean ATE by Repetition: {method}",
    width=400,
    height=300
)

/var/folders/96/010vljmj6rlcmhk35tq09c2d4r043f/T/ipykernel_71811/3595517540.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df = D.groupby("size_bin").agg(


alt.Chart(...)

In [92]:
method = "min_k"
D = blocks.merge(noblocks, on=["doc_id", "method"])[["blocks_score", "size", "noblocks_score", "doc_id", "method"]].drop_duplicates()
D["delta"] = D["blocks_score"] - D["noblocks_score"]
D = D[D["size"] <= 30].copy()
D["size_bin"] = pd.cut(D["size"], bins=range(0, 31, 3), right=True)
D = D[D['method'] == method].copy()

df = D.groupby("size_bin").agg(
    delta_mean=("delta", "mean"),
    count=("delta", "count")
).reset_index()

import altair as alt

df["size_bin"] = df["size_bin"].astype(str)
alt.Chart(df).mark_bar().encode(
    x=alt.X("size_bin:N", sort=None, title="Count of similar documents in BlockBench"),
    y=alt.Y("delta_mean:Q", title="Mean ATE"),
    tooltip=["size_bin", "delta_mean", "count"]
).properties(
    title=f"Mean ATE by Repetition: {method}",
    width=400,
    height=300
)

/var/folders/96/010vljmj6rlcmhk35tq09c2d4r043f/T/ipykernel_71811/578058848.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df = D.groupby("size_bin").agg(


alt.Chart(...)